# Predicting term-deposit subscriptions

**Question:** Can we predict whether a client subscribes to a term deposit (`y`) after a marketing call?

This notebook builds a reproducible binary-classification experiment. Because only about 12% of clients subscribe, accuracy alone would be misleading; the primary metric is **average precision (PR-AUC)**, with ROC-AUC and threshold-dependent metrics reported as supporting measures.

Two realistic feature sets are evaluated:

- **After-call:** all predictors, including call `duration`. This answers the question as stated.
- **Before-call:** excludes `duration`, which is unavailable before the call and must not be used for pre-call targeting.

A held-out test set is touched only after model and threshold selection.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    average_precision_score, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42
DATA_PATH = Path('data.csv')
assert DATA_PATH.exists(), f'{DATA_PATH} not found; run this notebook beside data.csv'

In [ ]:
df = pd.read_csv(DATA_PATH)
display(df.head())
print(f'Rows: {len(df):,} | predictors: {df.shape[1] - 1}')
print('Duplicate rows:', df.duplicated().sum())
print('Target values:', sorted(df['y'].dropna().unique()))

summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_n': df.isna().sum(),
    'missing_pct': (100 * df.isna().mean()).round(1),
    'unique': df.nunique(dropna=True),
})
display(summary)

## Data checks and exploratory analysis

Missing categorical values are informative in this dataset (for example, no previous campaign outcome). The pipeline imputes them to an explicit `Missing` category. Numeric missing values, if any, are median-imputed. No rows are dropped.

In [ ]:
target_counts = df['y'].value_counts().rename_axis('y').to_frame('count')
target_counts['percent'] = (100 * target_counts['count'] / len(df)).round(2)
display(target_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='y', hue='y', legend=False, ax=axes[0])
axes[0].set(title='Imbalanced target', xlabel='Subscribed')
sns.boxplot(data=df, x='y', y='duration', hue='y', legend=False, showfliers=False, ax=axes[1])
axes[1].set(title='Call duration by outcome', xlabel='Subscribed', ylabel='Duration (seconds)')
plt.tight_layout()
plt.show()

rate_by_month = (df.assign(subscribed=df['y'].eq('yes'))
                   .groupby('month', observed=True)['subscribed']
                   .agg(['mean', 'size'])
                   .sort_values('mean', ascending=False))
rate_by_month['mean'] = (100 * rate_by_month['mean']).round(1)
display(rate_by_month.rename(columns={'mean': 'subscription_pct', 'size': 'calls'}))

## Split first, then preprocess

The 70/15/15 train/validation/test split is stratified. All imputation, scaling, and one-hot encoding are learned inside each training fold, preventing preprocessing leakage. The validation set will be used only to select a probability threshold; the test set remains the final unbiased check.

In [ ]:
X = df.drop(columns='y')
y = df['y'].map({'no': 0, 'yes': 1})
assert y.notna().all(), 'Unexpected target label found'

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)
split_summary = pd.DataFrame({
    'rows': [len(y_train), len(y_valid), len(y_test)],
    'positive_rate': [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=['train', 'validation', 'test'])
display(split_summary.style.format({'positive_rate': '{:.2%}'}))

def make_pipeline(columns, model):
    categorical = X_train[columns].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    numeric = [c for c in columns if c not in categorical]
    preprocessor = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical),
    ])
    return Pipeline([('preprocess', preprocessor), ('model', model)])

after_call_features = X.columns.tolist()
before_call_features = [c for c in X.columns if c != 'duration']

## Cross-validated model comparison

A regularized logistic regression is a strong, interpretable baseline for mixed tabular data. It is compared against a prevalence-only dummy model. `class_weight='balanced'` is also tested: it can improve minority recall, but ranking metrics determine whether it actually adds predictive information.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'pr_auc': 'average_precision', 'roc_auc': 'roc_auc', 'f1': 'f1'}
experiments = {
    'Dummy / after-call': (after_call_features, DummyClassifier(strategy='prior')),
    'Logistic / before-call': (before_call_features, LogisticRegression(max_iter=1500, C=1.0)),
    'Logistic / after-call': (after_call_features, LogisticRegression(max_iter=1500, C=1.0)),
    'Balanced logistic / after-call': (after_call_features, LogisticRegression(max_iter=1500, C=1.0, class_weight='balanced')),
}

rows = []
for name, (features, estimator) in experiments.items():
    scores = cross_validate(
        make_pipeline(features, estimator), X_train[features], y_train,
        cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False,
    )
    row = {'experiment': name}
    for metric in scoring:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    rows.append(row)

cv_results = pd.DataFrame(rows).set_index('experiment').sort_values('pr_auc_mean', ascending=False)
display(cv_results.style.format('{:.3f}'))
print(cv_results.round(3).to_string())

## Select an operating threshold on validation data

The best cross-validated after-call logistic variant is refit on the training split. A default threshold of 0.5 is rarely optimal under class imbalance. For demonstration, the threshold that maximizes validation F1 is selected. In production this should instead reflect the relative business cost of a missed subscriber versus an unnecessary follow-up.

In [ ]:
candidate_names = ['Logistic / after-call', 'Balanced logistic / after-call']
selected_name = cv_results.loc[candidate_names, 'pr_auc_mean'].idxmax()
selected_features, selected_estimator = experiments[selected_name]
model = make_pipeline(selected_features, selected_estimator)
model.fit(X_train[selected_features], y_train)

valid_probability = model.predict_proba(X_valid[selected_features])[:, 1]
precision, recall, thresholds = precision_recall_curve(y_valid, valid_probability)
f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_index = int(np.nanargmax(f1_values))
chosen_threshold = float(thresholds[best_index])

print('Selected model:', selected_name)
print(f'Validation PR-AUC: {average_precision_score(y_valid, valid_probability):.3f}')
print(f'Validation-selected F1 threshold: {chosen_threshold:.3f}')
print(f'Precision={precision[best_index]:.3f}, recall={recall[best_index]:.3f}, F1={f1_values[best_index]:.3f}')

## Final held-out test evaluation

In [ ]:
test_probability = model.predict_proba(X_test[selected_features])[:, 1]
test_prediction = (test_probability >= chosen_threshold).astype(int)
test_metrics = pd.Series({
    'PR-AUC (average precision)': average_precision_score(y_test, test_probability),
    'ROC-AUC': roc_auc_score(y_test, test_probability),
    'F1 at validation-selected threshold': f1_score(y_test, test_prediction),
    'Selected threshold': chosen_threshold,
})
display(test_metrics.to_frame('value').style.format('{:.3f}'))
print(test_metrics.round(3).to_string())
print(classification_report(y_test, test_prediction, target_names=['no', 'yes'], digits=3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
PrecisionRecallDisplay.from_predictions(
    y_test, test_probability, plot_chance_level=True,
    chance_level_kw={'color': 'grey'}, ax=axes[0]
)
axes[0].set_title('Test precision–recall curve')
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[1])
axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set_title('Test ROC curve')
ConfusionMatrixDisplay.from_predictions(y_test, test_prediction, display_labels=['no', 'yes'], cmap='Blues', ax=axes[2])
axes[2].set_title(f'Confusion matrix (threshold={chosen_threshold:.2f})')
plt.tight_layout()
plt.show()

## Interpretation

For the logistic model, coefficients describe associations on the log-odds scale after numeric standardization. They are useful for directional interpretation, not causal claims. Correlated predictors can share or distort coefficient magnitude.

In [ ]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = pd.Series(
    model.named_steps['model'].coef_.ravel(), index=feature_names, name='coefficient'
)
most_influential = pd.concat([coefficients.nsmallest(10), coefficients.nlargest(10)]).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = np.where(most_influential < 0, '#C44E52', '#4C72B0')
most_influential.plot.barh(color=colors, ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Largest logistic-regression coefficients', xlabel='Coefficient', ylabel='Encoded feature')
plt.tight_layout()
plt.show()
display(most_influential.to_frame())

## Conclusion and limitations

Use the executed results above to answer the question: prediction is useful only if the after-call model materially exceeds the dummy model's PR-AUC (approximately the positive prevalence), performs consistently across folds, and retains that performance on the held-out test set. The before-call result quantifies how much signal remains without call duration.

Important limitations:

- `duration` makes the after-call model unsuitable for deciding whom to call; use the before-call model for that purpose.
- This is observational historical data. Coefficients and subgroup conversion rates are not causal effects.
- Random splitting assumes future cases resemble the same mixture. If timestamps or campaign batches become available, an out-of-time test is preferable.
- Threshold selection should ultimately use business costs, contact capacity, and calibration checks.
- Before deployment, assess subgroup performance/fairness, drift, and whether outreach based on predictions complies with applicable rules.